# Crossing the Aisle: results and checks

**Question:** when a change of government makes parties vote together, do their members' floor challenges and language follow?

This notebook reads the outputs of `tools/run_all.py`. See `AI_METHODS.md` for what each measure captures and misses. Hand labels may still be pending; the last section says.

In [1]:
import json
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "data" / "results"
read = lambda name: json.loads((RESULTS / f"{name}.json").read_text(encoding="utf-8"))
network, language, role, votes, summary, validation, labels = map(read, ["network", "language", "role", "votes", "summary", "validation", "label_checks"])
pd.set_option("display.width", 180, "display.max_columns", 30, "display.max_rows", 80)
pd.DataFrame(network["windows"])

,name,from,to,government
0,2009 KF+V,2009-11-24,2011-10-03,"[KF, V]"
1,2011 RV+S+SF,2011-10-03,2014-02-03,"[RV, S, SF]"
2,2014 RV+S,2014-02-03,2015-06-28,"[RV, S]"
3,2015 V,2015-06-28,2016-11-28,[V]
4,2016 KF+LA+V,2016-11-28,2019-06-27,"[KF, LA, V]"
5,2019 S,2019-06-27,2022-12-15,[S]
6,2022 M+S+V,2022-12-15,2026-06-03,"[M, S, V]"
7,2026 M+RV+S+SF,2026-06-03,9999-12-31,"[M, RV, S, SF]"


## 1. Votes set up the question

Rebel votes are rare, and vote network communities match party about as well after shuffling members' votes within their party. Member votes add nothing beyond party.

In [2]:
pd.DataFrame(votes["rebels"])

,window,cast,rebel_votes,rebel_rate,votes,votes_with_a_rebel,absent_share
0,2016 KF+LA+V,97922,55,0.00056,945,0.044,0.414
1,2019 S,153956,211,0.00137,1689,0.083,0.448
2,2022 M+S+V,153931,209,0.00136,1467,0.100,0.398
3,2026 M+RV+S+SF,4405,15,0.00341,41,0.171,0.377


In [3]:
pd.DataFrame(votes["vote_network"]).T.drop(columns="community_parties")

,members,contested_votes,communities,modularity,seed_stability_nmi,nmi_with_party,null_within_party_shuffle_nmi
2016 KF+LA+V,191,658,3,0.368,0.969,0.678,0.678
2019 S,203,1350,4,0.476,1.0,0.71,0.746
2022 M+S+V,210,1200,3,0.225,0.992,0.6,0.669


## 2. The challenge network

Each row is one period between government formations. Compare modularity with the rewired null, and assortativity with the label shuffle.

In [4]:
rows = []
for name, w in network["by_window"].items():
    rows.append({"window": name, **w["summary"],
                 "party_r": w["party_assortativity"]["observed"], "party_null": w["party_assortativity"]["null_mean"],
                 "side_r": w["side_assortativity"]["observed"], "side_null": w["side_assortativity"]["null_mean"],
                 "Q": w["communities"]["modularity"], "Q_null": w["communities"]["null_modularity_mean"],
                 "seed_nmi": w["communities"]["seed_stability_nmi_mean"], "party_nmi": w["communities"]["nmi_with_party"],
                 "share_committee": w["committee_test"]["share_sharing_a_committee"], "committee_null": w["committee_test"]["null_mean"]})
pd.DataFrame(rows).set_index("window")

,members,challenges,distinct_pairs,density,giant_weak_component,made_median,made_max,received_median,received_max,party_r,party_null,side_r,side_null,Q,Q_null,seed_nmi,party_nmi,share_committee,committee_null
window,,,,,,,,,,,,,,,,,,,
2009 KF+V,194,8193,3220,0.0860,194,24.5,469,32.0,203,-0.1470,-0.0075,-0.3256,-0.0081,0.3516,0.0774,0.856,0.078,0.546,0.410
2011 RV+S+SF,198,7901,3429,0.0879,198,21.5,302,28.0,222,-0.1372,-0.0065,-0.3599,-0.0061,0.3548,0.0790,0.848,0.057,0.746,0.700
2014 RV+S,191,3535,1963,0.0541,191,10.0,219,14.0,91,-0.1315,-0.0080,-0.2527,-0.0068,0.3973,0.1224,0.898,0.066,0.661,0.585
2015 V,193,3138,1781,0.0481,193,8.0,153,11.0,84,-0.1133,-0.0078,-0.1336,-0.0094,0.4167,0.1303,0.924,0.058,0.655,0.601
2016 KF+LA+V,210,4429,2289,0.0522,210,10.5,187,16.0,105,-0.1169,-0.0066,-0.2253,-0.0065,0.4506,0.1115,0.890,0.062,0.704,0.608
2019 S,224,7869,3964,0.0794,222,18.0,358,25.5,242,-0.1004,-0.0078,-0.1405,-0.0079,0.3412,0.0817,0.849,0.100,0.813,0.772
2022 M+S+V,222,8067,4191,0.0854,222,14.0,423,29.0,179,-0.0856,-0.0073,-0.2895,-0.0075,0.2771,0.0812,0.730,0.083,0.830,0.787
2026 M+RV+S+SF,93,204,176,0.0206,93,1.0,21,2.0,11,-0.0974,-0.0095,-0.2619,-0.0080,0.5430,0.4375,0.918,0.221,0.603,0.447


## 3. The main test: what changes at a formation

`mean_effect` is in units of the spread of pairs whose relation did not change at that formation. Rows have different raw units, so compare each with its placebo rather than with each other.

In [5]:
layers = {**summary["layers"], **summary["challenge_variants"]}
pd.DataFrame({k: {x: v[x] for x in ["pairs_tested", "right_direction", "mean_effect_in_placebo_sd", "median_effect", "p_one_sided", "formations_right_on_average"]} for k, v in layers.items()}).T

,pairs_tested,right_direction,mean_effect_in_placebo_sd,median_effect,p_one_sided,formations_right_on_average
votes,14.0,13.0,2.43,2.27,0.000050,6.0
challenges,14.0,14.0,8.42,7.38,0.000050,6.0
same_debate_similarity,14.0,13.0,3.65,3.01,0.000050,6.0
party_classifier,8.0,3.0,0.06,-0.11,0.119094,3.0
challenges_backbench_only,14.0,14.0,10.24,8.76,0.000050,6.0
challenges_same_members_panel,10.0,10.0,9.78,9.34,0.000050,6.0


In [6]:
focal = pd.DataFrame(network["transitions_all"])
focal = focal[focal["relation"].isin(["joined", "both_entered", "split"])]
focal[["transition", "pair", "relation", "observed_before", "expected_before", "observed_after", "expected_after", "log_ratio_change", "placebo_median"]]

,transition,pair,relation,observed_before,expected_before,observed_after,expected_after,log_ratio_change,placebo_median
17,2011-10-03,KF-V,split,20,211.0,40,139.4,1.094,0.084
22,2011-10-03,RV-S,both_entered,58,161.7,15,130.4,-1.113,0.084
23,2011-10-03,RV-SF,both_entered,30,112.7,12,52.4,-0.132,0.084
25,2011-10-03,S-SF,both_entered,45,293.3,7,164.2,-1.224,0.084
51,2014-02-03,RV-SF,split,12,52.4,44,61.0,1.119,0.001
53,2014-02-03,S-SF,split,7,164.2,113,135.4,2.909,0.001
78,2015-06-28,RV-S,split,6,68.4,17,31.4,1.763,-0.024
113,2016-11-28,KF-LA,both_entered,17,14.7,2,17.0,-2.087,-0.010
117,2016-11-28,KF-V,joined,6,28.2,2,26.3,-0.888,-0.010
121,2016-11-28,LA-V,joined,42,68.9,7,65.4,-1.682,-0.010


## 4. How fast: S and V around December 2022

In [7]:
t = network["timelines"]["S-V 2022-12-15"]
pd.DataFrame(t["votes"]).merge(pd.DataFrame(t["challenges"]), on="quarter")

,quarter,votes,agreement,challenges,observed,expected
0,-10,89,0.854,185,15.0,8.3
1,-9,102,0.794,765,76.0,55.3
2,-8,270,0.663,1011,87.0,67.5
3,-7,352,0.705,1408,117.0,90.6
4,-6,13,0.923,61,4.0,2.0
5,-5,83,0.795,564,44.0,33.5
6,-4,229,0.638,561,36.0,31.1
7,-3,301,0.681,1166,108.0,94.0
8,-2,4,1.000,55,10.0,4.9
9,-1,11,0.818,202,7.0,5.6


## 5. Language

Same-debate similarity against a within-debate shuffle, the party classifier against shuffled labels, and the role classifier on a government it did not see. S–V similarity had been rising long before 2022.

In [8]:
pd.DataFrame([r for r in language["same_debate"] if r["pair"] == "S-V"])

,window,pair,debates,similarity,null,excess,se
0,2009 KF+V,S-V,650,0.7081,0.7066,0.0015,0.0043
1,2011 RV+S+SF,S-V,572,0.7318,0.7128,0.0190,0.0045
2,2014 RV+S,S-V,578,0.7489,0.7234,0.0255,0.0040
3,2015 V,S-V,512,0.7538,0.7161,0.0377,0.0049
4,2016 KF+LA+V,S-V,602,0.7531,0.7170,0.0362,0.0041
5,2019 S,S-V,583,0.7458,0.7002,0.0456,0.0041
6,2022 M+S+V,S-V,499,0.7659,0.7124,0.0535,0.0047
7,2026 M+RV+S+SF,S-V,36,0.7452,0.6981,0.0472,0.0162


In [9]:
pd.DataFrame([r for r in language["accuracy"] if r["pair"] == "S-V"])

,window,pair,speakers_a,speakers_b,chunks,accuracy,null_mean,null_max
0,2009 KF+V,S-V,47,43,1593,0.718,0.486,0.538
1,2011 RV+S+SF,S-V,38,49,1602,0.693,0.482,0.534
2,2014 RV+S,S-V,37,47,1465,0.661,0.497,0.559
3,2015 V,S-V,52,22,1286,0.528,0.495,0.530
4,2016 KF+LA+V,S-V,48,29,1455,0.584,0.482,0.541
5,2019 S,S-V,40,47,1613,0.627,0.489,0.532
6,2022 M+S+V,S-V,49,24,1317,0.525,0.493,0.543
7,2026 M+RV+S+SF,S-V,17,14,274,0.425,0.462,0.589


In [10]:
print(role["within_party"]["mean_gap"], "+/-", role["within_party"]["null_sd"], "p =", role["within_party"]["p"])
display(pd.DataFrame(role["within_party"]["parties"]))
pd.DataFrame(role["role_classifier"]).drop(columns=["opposition_words", "government_words"])

3.92 +/- 0.32 p = 0.0001999600079984003


,party,opposition_rate,government_rate,quarters
0,KF,5.83,3.49,67
1,LA,6.22,3.45,67
2,RV,5.54,3.55,65
3,S,10.79,4.27,68
4,SF,5.80,4.22,68
5,V,11.98,3.64,67


,window,test_chunks,train_chunks,accuracy,null_mean,null_max
0,2009 KF+V,3451,14476,0.657,0.494,0.523
1,2011 RV+S+SF,3436,10918,0.642,0.499,0.529
2,2014 RV+S,3262,10903,0.660,0.502,0.529
3,2015 V,3374,10411,0.659,0.499,0.560
4,2016 KF+LA+V,3686,9935,0.658,0.505,0.569
5,2019 S,4058,10360,0.638,0.491,0.534
6,2022 M+S+V,4011,13822,0.663,0.498,0.533


## 6. Hand validation

**Read these.** First, reproduce the observed S→V count for one sitting by reading the transcript turns printed below. Count the lines yourself and compare with the number from the pipeline.

In [11]:
speeches = pd.read_parquet(ROOT / "data-raw" / "derived" / "speeches.parquet",
                           columns=["session", "meeting", "turn", "first", "last", "party", "role", "is_chair", "text"])
edges = pd.read_parquet(ROOT / "data" / "challenges.parquet")
pair = edges[edges["floor_party"].isin(["S", "V"]) & edges["challenger_party"].isin(["S", "V"]) & (edges["floor_party"] != edges["challenger_party"])]
session, meeting = pair.groupby(["session", "meeting"]).size().sort_values(ascending=False).index[3]
sitting = speeches[(speeches["session"] == session) & (speeches["meeting"] == meeting)].reset_index(drop=True)
found = pair[(pair["session"] == session) & (pair["meeting"] == meeting)]
print(f"Session {session}, sitting {meeting}: the pipeline finds {len(found)} S-V challenges. Each is shown as floor / chair / challenger:\n")
for turn in found["turn"]:
    i = sitting.index[sitting["turn"] == turn][0]
    for row in sitting.iloc[i - 2:i + 1].itertuples():
        print(f"  [{row.party}] {row.first} {row.last}: {row.text[:150]}")
    print()

Session 20111, sitting 18: the pipeline finds 36 S-V challenges. Each is shown as floor / chair / challenger:

  [V] Ulla Tørnæs: Jeg er ikke et sekund i tvivl om, at det faktisk vil blive vanskeligere for hr. Per Clausens gode ven at få fastholdelse på arbejdsmarkedet. For ud ov
  [S] Mogens Lykketoft: Hr. Rasmus Prehn for en kort bemærkning.
  [S] Rasmus Prehn: Jeg synes, at det var meget slående, at Venstres ordfører slet ikke nævnte børn i forbindelse med den her sag. Sagen er jo den, at når den nye regerin

  [V] Ulla Tørnæs: Mishag? Hvem sagde, at det var mishag? (Munterhed). Jeg er nødt til at sige til hr. Rasmus Prehn, at jeg overhovedet ikke kan genkende det billede, hr
  [S] Mogens Lykketoft: Hr. Benny Engelbrecht for en kort bemærkning.
  [S] Benny Engelbrecht: Jeg vil godt stille et helt enkelt spørgsmål: Hvor mange børnefamilier synes Venstres ordfører, at det er i orden lever i fattigdom i Danmark? Kan vi 

  [V] Ulla Tørnæs: Jeg kan sige til hr. Benny Engelbrecht: Vi øns

Second, the agent's automated checks. These catch code bugs; they do not show that a measure means what we claim.

In [12]:
pd.DataFrame(validation["checks"])

,check,passed,detail
0,toy: observed party matrix,True,"[[0.0, 3.0], [2.0, 0.0]]"
1,toy: null spreads edges across cells,True,"[[0.6, 1.79], [0.79, 0.6]]"
2,assortativity matches networkx,True,-0.0526 vs -0.0526
3,status: S+V into government together,True,
4,status: pair leaves together,True,
5,status: both outside before and after,True,
6,vote parser: new format,True,"{'for': {'S', 'SF'}, 'imod': {'V'}, 'hverken':..."
7,vote parser: old format,True,"{'for': {'S', 'RV'}, 'imod': {'DF', 'V'}, 'hve..."
8,vote parser: list ending in og,True,"{'for': {'M', 'S', 'V'}, 'imod': {'SF', 'DD'},..."
9,vote parser: named member with initial,True,"{'for': {'NB', 'SF', 'LA'}, 'imod': {'M', 'S',..."


In [13]:
validation["recounts"]

{'s_v_challenges': {'raw_xml': {'before': 321, 'after': 10},
  'pipeline': {'before': 321, 'after': 8}},
 's_v_votes': {'from_member_votes': 1.0,
  'votes': 1467,
  'from_summaries': 1.0,
  'summaries_n': 1557}}

## 7. Hand labels

Agreement with the group's labels. `not labeled yet` means the result is still unvalidated, and the site says so.

In [14]:
labels

{'challenge_precision': {'status': 'not labeled yet'},
 'challenge_missed': {'status': 'not labeled yet'},
 'name_resolution': {'status': 'not labeled yet'},
 'challenge_stance': {'status': 'not labeled yet'}}